# Quantum-native workflow: measure once, render many

**Moth Hack 2026 — Challenge 10 (Quantum-native 2)**

This notebook demonstrates a full workflow on Moth's `retrocausal-echo-v1` engine:

1. Submit one job with **no audio input** — the engine measures a quantum impulse response (IR) on Aer and hands back the raw tap map plus a reference render of the IR itself.
2. Inspect that tap map: what the quantum measurement actually produced, in numbers.
3. Reuse the same measured IR to render a real audio clip — **with no further quantum work**, since Moth's own docs confirm passing a saved `ir` asset id skips re-measurement entirely.

The parameters used here (`theta_x=2.6`, `theta_zz≈0.25π`, `negative_mode="invert"`) came out of a small sweep across `theta_x` (see `retrocausal_pipeline.py sweep` in the accompanying repo) — this was the point that sounded most distinctly "quantum" rather than like an ordinary delay, deep in the erasure/inversion regime the engine's own docs describe.

## 0. Setup

Requires `MOTH_API_KEY` in the environment. See the repo's `retrocausal_pipeline.py` for the same logic factored into a reusable CLI — this notebook inlines it for readability.

In [ ]:
import json
import os
import time
from pathlib import Path

import requests

API = "https://api.mothquantum.com/api/v1"
ENGINE = "retrocausal-echo-v1"
H = {
    "Authorization": f"Bearer {os.environ['MOTH_API_KEY']}",
    # Cloudflare 403s a bare Python-urllib user agent — identify the client.
    "User-Agent": "moth-hack-challenge-10-notebook/1.0",
}
OUT = Path("notebook_output")
OUT.mkdir(exist_ok=True)


def check(resp):
    if not resp.ok:
        raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:800]}")
    return resp.json()


def run_job(params, input_files=None):
    """Submit a job, poll to completion, return (job_id, result_body)."""
    job_id = check(requests.post(
        f"{API}/engines/{ENGINE}/process", headers=H,
        json={"params": params, "input_files": input_files or {}},
    ))["job_id"]

    while True:
        status = check(requests.get(f"{API}/jobs/{job_id}/status", headers=H))
        if status["status"] in ("completed", "failed", "cancelled"):
            break
        time.sleep(2)

    if status["status"] != "completed":
        err = (status.get("error") or {}).get("message")
        raise RuntimeError(f"job {job_id} ended {status['status']}: {err}")

    return job_id, check(requests.get(f"{API}/jobs/{job_id}/result", headers=H))


def outputs_by_slot(result):
    return {o["slot"]: o for o in (result.get("outputs") or [])}


def asset_id_of(output_entry):
    # The generic API docs and this engine's own code sample disagree on the
    # field name (output_asset_id vs asset_id) — accept either.
    aid = output_entry.get("asset_id") or output_entry.get("output_asset_id")
    if not aid:
        raise KeyError(f"no asset id in output entry: {output_entry}")
    return aid


def download(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    r = requests.get(url)
    r.raise_for_status()
    dest.write_bytes(r.content)


def upload_asset(path, content_type):
    size = path.stat().st_size
    reg = check(requests.post(
        f"{API}/assets", headers=H,
        json={"filename": path.name, "content_type": content_type, "size_bytes": size},
    ))
    up = reg["upload"]
    with open(path, "rb") as fh:
        r = requests.put(up["url"], data=fh, headers=up["headers"])
        if not r.ok:
            raise RuntimeError(f"upload HTTP {r.status_code}: {r.text[:400]}")
    check(requests.post(f"{API}/assets/{reg['asset_id']}/complete", headers=H))
    return reg["asset_id"]

## 1. Measure a quantum impulse response

No `audio` input, so the engine calls `quantum_echo.measure_ir` in-process, on Aer (local, noiseless simulation, no IBM token needed), and hands back:

- `result` — a reference stereo WAV of the IR itself
- `ir` — the measured impulse response as a portable `trajectory` JSON envelope
- `taps` — the full tap map: time, level, pan, complex amplitude and polarity per tap

`theta_x` is the "how quantum" dial: low values give regular, delay-like taps; high values push into an erasure/inversion regime where the echo stops behaving like a classical delay. `theta_zz` sets the coupling between sites — ~0.25π is the engine's own documented "sparse" setting.

In [ ]:
import math

MEASURE_PARAMS = {
    "theta_x": 2.6,
    "theta_zz": 0.25 * math.pi,
    "negative_mode": "invert",
    "n_sites": 8,
    "depth": 8,
    "ir_seconds": 4,
}

job_id, result = run_job(MEASURE_PARAMS)
outs = outputs_by_slot(result)
ir_asset_id = asset_id_of(outs["ir"])

download(outs["result"]["url"], OUT / "ir_reference.wav")
download(outs["ir"]["url"], OUT / "ir.json")
download(outs["taps"]["url"], OUT / "taps.json")

print(f"job_id      = {job_id}")
print(f"ir_asset_id = {ir_asset_id}")
print(f"saved to    = {OUT}/")

## 2. Read the tap map

This is where the quantum measurement becomes numbers. Each tap is one measured event: when it lands (`time_ms`), how loud (`level`), where in the stereo field (`pan`), the complex amplitude `F` the circuit produced, and its `polarity`. A negative `F` is what `negative_mode` acts on — here, `invert` flips its polarity into a phase-cancelling echo.

In [ ]:
taps_body = json.loads((OUT / "taps.json").read_text())
tap_map = taps_body["extras"]["tap_map"]["taps"]

print(f"{len(tap_map)} taps survived min_level at theta_x={MEASURE_PARAMS['theta_x']}\n")
for t in tap_map[:8]:
    print(
        f"  t={t['time_ms']:>7.1f}ms  level={t['level']:.3f}  "
        f"pan={t['pan']:+.2f}  F={t['F']}  polarity={t['polarity']}"
    )
if len(tap_map) > 8:
    print(f"  ... and {len(tap_map) - 8} more")

## 3. Reuse the measurement — render real audio, no new quantum work

Passing the saved `ir_asset_id` back in as the `ir` input file makes the engine call `EchoIR.from_trajectory` instead of re-measuring. Per Moth's own engine description:

> *"Re-render a measured response with different audio, tempo, polarity behaviour or feedback without paying for the measurement again."*

So this cell is the same measured tap map from Section 1, just pointed at a real audio file. Every echo parameter (`theta_x`, `theta_zz`, `depth`, ...) is now ignored — only mix/feedback/tempo params matter here.

In [ ]:
SOURCE_AUDIO = Path("Cyberwaste_trimmed.wav")  # swap in your own clip

audio_asset_id = upload_asset(SOURCE_AUDIO, "audio/wav")

render_job_id, render_result = run_job(
    {"mix": 0.7, "feedback": 0.4},
    input_files={"audio": audio_asset_id, "ir": ir_asset_id},
)
render_outs = outputs_by_slot(render_result)

rendered_path = OUT / f"rendered_{SOURCE_AUDIO.stem}.wav"
download(render_outs["result"]["url"], rendered_path)

print(f"render job_id = {render_job_id}")
print(f"rendered file = {rendered_path}")

## 4. Listen

In [ ]:
from IPython.display import Audio, display

print("Reference IR alone (no input audio, the effect's own impulse):")
display(Audio(str(OUT / "ir_reference.wav")))

print(f"\n{SOURCE_AUDIO.name} rendered through the measured IR:")
display(Audio(str(rendered_path)))

## Why this matters

The expensive part — the quantum measurement — happened exactly once, in Section 1. Sections 3–4 can be repeated indefinitely against any audio, any mix setting, any tempo, at zero additional quantum cost, because the tap map is portable data (`ir.json`), not a live circuit. That's the same pattern behind the accompanying repo's `sweep` command: measure a small grid of `theta_x` points once, then re-render everything against them as many times as needed.

It's also the fix for why a naive "call the quantum API per audio event" design doesn't work in practice — Moth's own API is asynchronous, poll-based, with no sub-second guarantee. Baking the quantum work into a reusable asset ahead of time is what makes this usable in something like a plugin or sampler, rather than something you can only run once and wait for.